In [0]:
%sql
SELECT
  ROUND(CORR(mensual.tm, mensual.lluvia), 3) AS correlacion_venta_lluvia
FROM (
  SELECT f.departamento_destino, year(f.fecha_pedido) AS anio, month(f.fecha_pedido) AS mes,
         SUM(f.tm_vendidas) AS tm,
         AVG(c.precip_real_mm) AS lluvia
  FROM workspace.gold.fact_ventas f
  JOIN workspace.bronze.clima_raw c
    ON f.departamento_destino = c.departamento
   AND year(f.fecha_pedido)  = c.anio
   AND month(f.fecha_pedido) = c.mes
  GROUP BY f.departamento_destino, year(f.fecha_pedido), month(f.fecha_pedido)
) AS mensual;

In [0]:
%sql
SELECT
  year(fecha_pedido) AS anio,
  ROUND(SUM(GREATEST(tm_programadas - tm_vendidas, 0)), 0) AS sobre_programado_tm,
  ROUND(SUM(GREATEST(tm_vendidas - tm_programadas, 0)), 0) AS sub_programado_tm,
  ROUND(SUM(ABS(tm_programadas - tm_vendidas)), 0)         AS desajuste_total_tm
FROM workspace.gold.fact_ventas
GROUP BY year(fecha_pedido)
ORDER BY anio;

supuesto*

In [0]:
%sql
-- SUPUESTO declarado: costo de almacenamiento de materia prima sobrante
-- 2 USD por tonelada por mes (valor de referencia, ajustable)
SELECT
  year(fecha_pedido) AS anio,
  ROUND(SUM(GREATEST(tm_programadas - tm_vendidas, 0)), 0)        AS sobre_stock_tm,
  ROUND(SUM(GREATEST(tm_programadas - tm_vendidas, 0)) * 2, 0)    AS costo_sobre_stock_usd
FROM workspace.gold.fact_ventas
GROUP BY year(fecha_pedido)
ORDER BY anio;

In [0]:
%sql
SELECT
  p.nombre,
  p.vida_util_dias,
  p.sensible_humedad,
  ROUND(SUM(GREATEST(f.tm_programadas - f.tm_vendidas, 0)), 0) AS sobre_stock_tm
FROM workspace.gold.fact_ventas f
JOIN workspace.gold.dim_producto p ON f.producto_id = p.producto_id
GROUP BY p.nombre, p.vida_util_dias, p.sensible_humedad
ORDER BY sobre_stock_tm DESC;